In [1]:
# ==========================================================
# Cell 1 : Import Libraries
# ==========================================================

import os
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

# Create output folder
os.makedirs("../data/processed", exist_ok=True)

# Load cleaned dataset
df = pd.read_csv("../data/processed/retail_clean.csv")

print("Dataset Loaded Successfully")
print("Shape:", df.shape)

display(df.head())

Dataset Loaded Successfully
Shape: (20000, 13)


,Order_ID,Order_Date,Customer_ID,Product_ID,Category,Region,Quantity,Unit_Price,Discount,Sub_Category,Sales,Cost,Profit
0,ORD100000,2024-05-10,CUST01487,PROD0029,Beauty,Central,7,29.35,0.26,Skincare,152.03,98.82,53.21
1,ORD100001,2024-11-10,CUST00042,PROD0026,Grocery,North,7,247.81,0.07,Snacks,1613.24,1048.61,564.63
2,ORD100002,2022-05-02,CUST01197,PROD0070,Beauty,East,1,365.99,0.09,Makeup,333.05,216.48,116.57
3,ORD100003,2023-04-12,CUST00679,PROD0011,Beauty,South,2,269.54,0.16,Haircare,452.83,294.34,158.49
4,ORD100004,2022-11-27,CUST01274,PROD0100,Grocery,West,5,163.63,0.13,Dairy,711.79,462.66,249.13


In [2]:
# ==========================================================
# Cell 2 : Customer-Product Matrix
# ==========================================================

interaction = (
    df.groupby(["Customer_ID", "Product_ID"])["Quantity"]
      .sum()
      .unstack(fill_value=0)
)

print("Interaction Matrix Shape:", interaction.shape)

display(interaction.head())

Interaction Matrix Shape: (1499, 119)


Product_ID,PROD0001,PROD0002,PROD0003,PROD0004,PROD0005,PROD0006,PROD0007,PROD0008,PROD0009,PROD0010,...,PROD0110,PROD0111,PROD0112,PROD0113,PROD0114,PROD0115,PROD0116,PROD0117,PROD0118,PROD0119
Customer_ID,,,,,,,,,,,,,,,,,,,,,
CUST00001,0,0,0,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CUST00002,0,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CUST00003,0,0,0,0,3,0,0,0,0,0,...,9,0,0,0,0,0,0,0,0,6
CUST00004,0,0,0,0,0,0,0,0,0,0,...,0,3,0,0,0,0,0,0,0,0
CUST00005,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,0,4


In [3]:
# ==========================================================
# Cell 3 : Cosine Similarity
# ==========================================================

item_similarity = cosine_similarity(interaction.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=interaction.columns,
    columns=interaction.columns
)

print("Similarity Matrix Created")

display(item_similarity_df.iloc[:5, :5])

Similarity Matrix Created


Product_ID,PROD0001,PROD0002,PROD0003,PROD0004,PROD0005
Product_ID,,,,,
PROD0001,1.000000,0.110487,0.055204,0.114564,0.080228
PROD0002,0.110487,1.000000,0.065605,0.089126,0.070869
PROD0003,0.055204,0.065605,1.000000,0.061261,0.047250
PROD0004,0.114564,0.089126,0.061261,1.000000,0.098707
PROD0005,0.080228,0.070869,0.047250,0.098707,1.000000


In [4]:
# ==========================================================
# Cell 4 : Recommend Similar Products
# ==========================================================

def recommend_products(product_id, top_n=5):

    if product_id not in item_similarity_df.columns:
        return []

    scores = item_similarity_df[product_id].sort_values(
        ascending=False
    )

    scores = scores.drop(product_id)

    return scores.head(top_n).index.tolist()

In [5]:
# ==========================================================
# Cell 5 : Example Product Recommendation
# ==========================================================

sample_product = interaction.columns[0]

recommendations = recommend_products(sample_product)

print("Selected Product :", sample_product)

print("\nRecommended Products")

for i, product in enumerate(recommendations, 1):
    print(f"{i}. {product}")

Selected Product : PROD0001

Recommended Products
1. PROD0055
2. PROD0110
3. PROD0056
4. PROD0108
5. PROD0086


In [6]:
# ==========================================================
# Cell 6 : Recommend Products for Customer
# ==========================================================

def recommend_for_customer(customer_id, top_n=5):

    if customer_id not in interaction.index:
        return []

    customer_history = interaction.loc[customer_id]

    top_product = customer_history.idxmax()

    return recommend_products(
        top_product,
        top_n
    )

In [7]:
# ==========================================================
# Cell 7 : Example Customer Recommendation
# ==========================================================

sample_customer = interaction.index[0]

recommendations = recommend_for_customer(sample_customer)

print("Customer :", sample_customer)

print("\nRecommended Products")

for i, product in enumerate(recommendations, 1):
    print(f"{i}. {product}")

Customer : CUST00001

Recommended Products
1. PROD0027
2. PROD0002
3. PROD0022
4. PROD0105
5. PROD0045


In [8]:
# ==========================================================
# Cell 8 : Save Similarity Matrix
# ==========================================================

item_similarity_df.to_csv(
    "../data/processed/item_similarity_matrix.csv"
)

print("Similarity Matrix Saved Successfully!")

Similarity Matrix Saved Successfully!


In [9]:
# ==========================================================
# Cell 9 : Top Similar Products
# ==========================================================

product = interaction.columns[0]

top10 = (
    item_similarity_df[product]
    .sort_values(ascending=False)
    .iloc[1:11]
)

display(top10)

Product_ID
PROD0055    0.150915
PROD0110    0.145486
PROD0056    0.133321
PROD0108    0.131251
PROD0086    0.126515
PROD0082    0.116990
PROD0081    0.116852
PROD0011    0.116343
PROD0043    0.115314
PROD0004    0.114564
Name: PROD0001, dtype: float64

In [10]:
# ==========================================================
# Cell 10 : Summary
# ==========================================================

print("=" * 60)
print("RECOMMENDATION SYSTEM COMPLETED")
print("=" * 60)

print("Customers :", interaction.shape[0])
print("Products  :", interaction.shape[1])

print("\nFiles Generated")

print("----------------------------")
print("../data/processed/item_similarity_matrix.csv")

print("\nRecommendation Engine Ready!")

RECOMMENDATION SYSTEM COMPLETED
Customers : 1499
Products  : 119

Files Generated
----------------------------
../data/processed/item_similarity_matrix.csv

Recommendation Engine Ready!
